# 0. 무엇이 달라졌는가: 제품, 설계, 실습 구분

원문: [Super-agent 설계 글](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20), [제품 페이지](https://www.deepseek.com/en/harness/).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 학습 목표와 범위
Python 함수·타입 힌트·기본 async 문법을 아는 독자를 위한 약 4~6시간 과정입니다. 처음에는 실행 결과를 예측하고, 셀을 실행한 뒤 assert가 확인하는 성질을 설명하세요.

X 원문 제목은 *Exploring Super-agent Design: Towards a More Autonomous Agent System*입니다. 표시된 영문은 중국어 원문의 기계 번역이며, 첫머리에서 **아키텍처 아이디어이지 기존 DeepSeek Harness 기능 소개가 아니다**라고 밝힙니다. 공식 제품 페이지는 공개 프리뷰·오픈소스·Cordis 기반 플러그인 확장을 소개합니다. 제품 페이지의 애니메이션은 시연용이며 실제 실행 증거가 아닙니다.

따라서 아래 예제는 Super-agent 설계를 배우는 Python 교육용 구현입니다. 공식 Python SDK나 DeepSeek Harness의 Python 포팅, 내부 Super-agent 전체 구현의 공개 여부를 입증하는 자료가 아닙니다.

## 네 가지 목표와 다섯 모듈
|목표|뜻|실습|
|---|---|---|
|Continuous|채널을 바꿔도 작업·맥락이 이어짐|Router/Worker, Thread 구독, 참조 회수|
|Productive|여러 작업과 새 입력을 함께 처리|격리된 Worker, 비동기 start/stop|
|Proactive|환경 사건을 선별해 먼저 처리|코드 필터 → 의미 판단, 조건부 결과 전달|
|Persistent|장시간 유지·복구|상태 저장, 실행환경 분리, 외부 Watchdog|

원문의 다섯 모듈은 ① 2계층 구조 ② 이벤트 트리거 ③ 전이중 다채널 상호작용 ④ compute/state/environment 분리 ⑤ Watchdog입니다. 플러그인 설계는 이들을 조합할 기반으로 별도 학습합니다.

## 기존 구조와 비교할 때 주의할 점
모든 기존 harness가 단일 동기 루프인 것은 아닙니다. LangGraph도 병렬 처리, 상태, interrupt를 지원합니다. 차이는 라이브러리 우열보다 **어떤 책임과 불변조건을 구조로 강제하는가**에 있습니다.

|항목|단일 세션 중심 예|이번 설계|
|---|---|---|
|사용자 역할|여러 세션의 맥락을 직접 관리|채널/작업을 통해 하나의 시스템과 대화|
|주 세션|판단·실행·응답을 모두 수행|Router는 인간 소통·배정만 수행|
|작업 맥락|여러 작업의 중간 결과가 섞임|Worker마다 격리|
|후속 사건|매번 중앙 모델이 재배정|구독된 Thread는 코드로 pass-through|
|오래 걸리는 도구|완료까지 대기|즉시 receipt, 결과는 별도 사건|

Router는 단일 복잡한 과제를 세세히 분해하는 Team Lead와 역할이 다릅니다. Worker 자체가 팀일 수는 있습니다. LangGraph graph를 작성했다는 사실만으로 이 구조가 보장되지는 않습니다.

## 학습 순서
01 루프 → 02 Router와 이벤트 → 03 문맥 → 04 비동기 → 05 플러그인 → 06 복구 → 07 통합 → 08 선택적 실제 모델.

In [1]:
import sys
from importlib.metadata import version
print(sys.version)
for name in ('langchain','langgraph','ipykernel'):
    print(name, version(name))
assert sys.version_info >= (3,12)

3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
langchain 1.4.9
langgraph 1.2.19
ipykernel 7.4.0


## 실험 수치를 읽는 법
원문 §7은 합성 메시지와 시간을 소비하는 가짜 코딩 도구를 사용한 예비 실험이며, 코딩 능력 실험이 아닙니다. 5회 실행에서 Super-agent p50/p95는 12.4/15.7분, 중립 프롬프트 단일 세션은 20.4/29.5분입니다. Pass-through 제거 시 지연은 비슷하지만 cache-read token은 23.2M → 31.8M입니다. 이 수치가 모든 업무의 우월성이나 비용 절감을 보장하지 않습니다. 우리 실습은 논문 실험 재현이 아니라 구조적 성질의 검증입니다.

In [2]:
baseline_tokens=23.2
without_passthrough=31.8
print(f'Pass-through 제거 시 cache-read 증가: {(without_passthrough/baseline_tokens-1):.1%}')
assert without_passthrough > baseline_tokens

Pass-through 제거 시 cache-read 증가: 37.1%


## 직접 바꿔 보기

당신의 업무에서 사용자 채널, Thread, 작업, Worker를 각각 하나씩 정의하세요. 같은 채팅방의 모든 메시지를 하나의 작업으로 묶으면 어떤 문제가 생길까요?

## 해설과 참고 답안

예: GitHub 댓글(채널), PR #42(Thread), 회귀 버그 수정(작업), 수정 담당 세션(Worker). Thread와 Worker는 다대다이며 동일하지 않습니다. 한 Thread에서 서로 다른 작업이 시작될 수 있으므로 재배정 경로도 필요합니다.